<a href="https://colab.research.google.com/github/apHub27/sveltetech-ai-internship/blob/main/10_house_price_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import joblib

# Reading the file that you uploaded manually in the folder
df = pd.read_csv('Bengaluru_House_Data.csv')
print("Data loaded successfully from your upload!")

# Rest of the standard cleaning code
df = df[['location', 'size', 'total_sqft', 'price']].dropna()
df['location'] = df['location'].apply(lambda x: x.strip())
location_stats = df['location'].value_counts()
df['location'] = df['location'].apply(lambda x: 'Other' if x in location_stats[location_stats <= 10] else x)
df['BHK'] = df['size'].apply(lambda x: int(x.split(' ')[0]) if isinstance(x, str) else 1)

def clean_sqft(x):
    try: return float(x)
    except:
        tokens = str(x).split('-')
        if len(tokens) == 2: return (float(tokens[0]) + float(tokens[1])) / 2
        return None

df['Area'] = df['total_sqft'].apply(clean_sqft)
df = df.dropna()

location_mapping = df.groupby('location')['price'].mean().to_dict()
df['Location_Score'] = df['location'].map(location_mapping)
unique_locations = sorted(df['location'].unique().tolist())

X = df[['Area', 'BHK', 'Location_Score']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

artifacts = {'model': model, 'location_mapping': location_mapping, 'unique_locations': unique_locations}
joblib.dump(artifacts, 'house_project_bundle.pkl')
print("✅ Done! Your model is trained and saved.")

Data loaded successfully from your upload!
✅ Done! Your model is trained and saved.


In [21]:
%%writefile app.py
import streamlit as st
import joblib
import numpy as np

# FIX 1: Loading the correct bundle that has both model AND locations
bundle = joblib.load('house_project_bundle.pkl')
model = bundle['model']
location_mapping = bundle['location_mapping']
unique_locations = bundle['unique_locations']

st.title("🏡 Bengaluru Real-Estate Price Predictor")
st.write("Trained on real-world Kaggle data to predict house prices!")

# FIX 2: Added the missing location dropdown here
selected_location = st.selectbox("Select Location/Area:", unique_locations)

# Input fields for area and bedrooms
area = st.number_input("Enter House Area (sq ft):", min_value=300, max_value=20000, value=1200)
bhk = st.number_input("Select BHK (Bedrooms):", min_value=1, max_value=10, value=2)

if 'predicted_price' not in st.session_state:
    st.session_state.predicted_price = None

if st.button("Predict Price"):
    # FIX 3: Fetching the numerical score for the selected location
    loc_score = location_mapping.get(selected_location, location_mapping['Other'])

    # Feeding all 3 inputs (Area, BHK, Location_Score) to the model
    user_input = np.array([[area, bhk, loc_score]])
    prediction = model.predict(user_input)[0]
    st.session_state.predicted_price = round(prediction, 2)

if st.session_state.predicted_price is not None:
    price = st.session_state.predicted_price
    if price >= 100:
        st.success(f"💰 Estimated Price in {selected_location}: ₹{round(price/100, 2)} Crore")
    else:
        st.success(f"💰 Estimated Price in {selected_location}: ₹{price} Lakh")


Overwriting app.py


In [22]:
# 1. Clear any running instances of streamlit
!pkill streamlit

# 2. Run streamlit safely in the background
import os
os.system("streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false &")

# 3. Generate the secure Google Colab link
from google.colab import output
colab_url = output.eval_js("google.colab.kernel.proxyPort(8501)")

print("🔗 Google Streamlit Link Yeh Hai 👇")
print(colab_url)


🔗 Google Streamlit Link Yeh Hai 👇
https://8501-m-s-kkb-use1c1-18megw66scakm-c.us-east1-1.prod.colab.dev
